# ChurnIQ — Telecom Customer Churn & Revenue Risk Analyzer

## Phase 6.5 — Champion Model Validation

### Objective

Validate the final XGBoost model using 5-Fold Cross Validation.

The goals are:

- Verify model stability
- Measure generalization performance
- Reduce dependence on a single train-validation split
- Build confidence before deployment and business analysis

---

### Champion Model

XGBoost

Threshold: 0.40

Validation ROC-AUC: 0.6767

# ChurnIQ — Telecom Customer Churn & Revenue Risk Analyzer

## Phase 6.5 — Champion Model Validation

### Objective

Validate the final XGBoost model using 5-Fold Cross Validation.

The goals are:

- Verify model stability
- Measure generalization performance
- Reduce dependence on a single train-validation split
- Build confidence before deployment and business analysis

---

### Champion Model

XGBoost

Threshold: 0.40

Validation ROC-AUC: 0.6767

In [3]:
# ==========================================
# IMPORT LIBRARIES
# ==========================================

import pandas as pd
import numpy as np

from sklearn.model_selection import (
    StratifiedKFold,
    cross_val_score
)

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline

from xgboost import XGBClassifier

print("Libraries loaded successfully.")

Libraries loaded successfully.


In [4]:
# ==========================================
# LOAD DATA
# ==========================================

df = pd.read_csv(
    "../data/processed/feature_engineered.csv"
)

print(df.shape)

(51047, 67)


C:\Users\Akshit\AppData\Local\Temp\ipykernel_18380\1420387941.py:5: DtypeWarning: Columns (66) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(


In [5]:
# ==========================================
# TARGET ENCODING
# ==========================================

df["Churn"] = df["Churn"].map({
    "No": 0,
    "Yes": 1
})

In [6]:
# ==========================================
# FINAL FEATURE SET
# ==========================================

columns_to_drop = [
    "CustomerID",
    "ServiceArea",
    "MarketCode",
    "AreaCode",
    "CreditRating",
    "RetentionCalls",
    "RetentionOffersAccepted",
    "MadeCallToRetentionTeam"
]

model_df = df.drop(
    columns=columns_to_drop
)

print(model_df.shape)

(51047, 59)


In [7]:
# ==========================================
# FEATURES & TARGET
# ==========================================

X = model_df.drop(
    columns=["Churn"]
)

y = model_df["Churn"]

print(X.shape)
print(y.shape)

(51047, 58)
(51047,)


In [8]:
# ==========================================
# FEATURE TYPES
# ==========================================

numerical_features = X.select_dtypes(
    exclude=["object"]
).columns.tolist()

categorical_features = X.select_dtypes(
    include=["object"]
).columns.tolist()

print("Numerical:", len(numerical_features))
print("Categorical:", len(categorical_features))

Numerical: 38
Categorical: 20


In [9]:
# ==========================================
# CLASS IMBALANCE
# ==========================================

negative_class = (y == 0).sum()
positive_class = (y == 1).sum()

scale_pos_weight = (
    negative_class / positive_class
)

print(
    round(scale_pos_weight, 2)
)

2.47


In [11]:
# ==========================================
# PREPROCESSOR
# ==========================================

preprocessor = ColumnTransformer(
    transformers=[
        (
            "categorical",
            OneHotEncoder(
                handle_unknown="ignore"
            ),
            categorical_features
        )
    ],
    remainder="passthrough"
)

In [12]:
# ==========================================
# CHAMPION XGBOOST
# ==========================================

xgb_pipeline = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        (
            "model",
            XGBClassifier(
                n_estimators=300,
                max_depth=5,
                learning_rate=0.05,
                subsample=0.8,
                colsample_bytree=0.8,
                scale_pos_weight=scale_pos_weight,
                random_state=42,
                eval_metric="logloss"
            )
        )
    ]
)

print("Champion pipeline ready.")

Champion pipeline ready.


In [13]:
# ==========================================
# 5-FOLD CROSS VALIDATION
# ==========================================

cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

roc_auc_scores = cross_val_score(
    xgb_pipeline,
    X,
    y,
    cv=cv,
    scoring="roc_auc",
    n_jobs=-1
)

roc_auc_scores

array([0.68000088, 0.67288505, 0.685631  , 0.68321172, 0.67836273])

In [14]:
# ==========================================
# CROSS VALIDATION SUMMARY
# ==========================================

print("Fold Scores:")

for i, score in enumerate(
    roc_auc_scores,
    start=1
):
    print(
        f"Fold {i}: {score:.4f}"
    )

print()

print(
    "Mean ROC-AUC:",
    round(
        roc_auc_scores.mean(),
        4
    )
)

print(
    "Std ROC-AUC:",
    round(
        roc_auc_scores.std(),
        4
    )
)

Fold Scores:
Fold 1: 0.6800
Fold 2: 0.6729
Fold 3: 0.6856
Fold 4: 0.6832
Fold 5: 0.6784

Mean ROC-AUC: 0.68
Std ROC-AUC: 0.0044
